# Quik SKU Pre-listing Success Prediction
## Logistic Regression — Final Model

**Approach:** Leave-One-Month-Out (LOMO) evaluation — train on 4 months, test on held-out month  
**Dataset:** Filtered (Availability > 70%), N = 791 SKUs across 5 months  
**Features:** 30 pre-listing features — no Availability, no ROS (future signals excluded)  
**Label:** SKU success = `max(ROS_m1, ROS_m2) > max(min(Bucket_P10_Peer_ROS, Vel_Floor), 1)`  


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

MONTHS_ORDERED = ['Jul25', 'Sep25', 'Oct25', 'Nov25', 'Dec25']
BACKTEST_FILES = {
    'Jul25': "Jul'25 Backtest.xlsx",
    'Sep25': "Sep'25 Backtest.xlsx",
    'Oct25': "Oct'25 Backtest.xlsx",
    'Nov25': "Nov'25 Backtest.xlsx",
    'Dec25': "Dec'25 Backtest.xlsx",
}
MONTH_COLORS = {
    'Jul25':'#e6194b','Sep25':'#3cb44b','Oct25':'#4363d8',
    'Nov25':'#f58231','Dec25':'#911eb4'
}
print('Imports done.')


## 2. Feature Definitions

In [ ]:
BASE_FEATURES = [
    'Vel\n Score', 'Vel Floor', 'Benchmark', 'Sat\n Score', 'Launch\n Score', 'CQ\n SCORE',
    'NMI\n Unit%', 'NMI\n SKU%', 'CH\n SCORE', 'Return\n Score', 'Mon\n Score', 'SP\n SCORE',
    'rtv_binary', 'vel_ratio', 'vel_ratio_sq', 'vel_gap', 'above_floor', 'launch_pct',
    'nmi_total', 'nmi_ratio', 'sat_x_vel', 'cq_x_launch', 'cq_sq', 'rtv_x_mon',
    'bench_x_sat', 'is_width', 'conc_enc', 'asp_enc', 'asp_log', 'subcat_p50mv',
]

FEAT_LABELS = {
    'Vel\n Score':'Vel Score', 'Vel Floor':'Vel Floor', 'Benchmark':'Benchmark',
    'Sat\n Score':'Sat Score', 'Launch\n Score':'Launch Score', 'CQ\n SCORE':'CQ Score',
    'NMI\n Unit%':'NMI Unit%', 'NMI\n SKU%':'NMI SKU%', 'CH\n SCORE':'CH Score',
    'Return\n Score':'Return Score', 'Mon\n Score':'Mon Score', 'SP\n SCORE':'SP Score',
    'rtv_binary':'RTV Binary', 'vel_ratio':'Vel Ratio', 'vel_ratio_sq':'Vel Ratio²',
    'vel_gap':'Vel Gap', 'above_floor':'Above Floor', 'launch_pct':'Launch Pct',
    'nmi_total':'NMI Total', 'nmi_ratio':'NMI Ratio', 'sat_x_vel':'Sat×Vel',
    'cq_x_launch':'CQ×Launch', 'cq_sq':'CQ²', 'rtv_x_mon':'RTV×Mon',
    'bench_x_sat':'Bench×Sat', 'is_width':'Is Width', 'conc_enc':'Conc Level',
    'asp_enc':'ASP Bucket', 'asp_log':'ASP (log)', 'subcat_p50mv':'Subcat P50 MV',
}

print(f'{len(BASE_FEATURES)} features defined.')


## 3. Data Loading & Feature Engineering

In [ ]:
def load_month(path, month):
    """Load one backtest month and engineer all features."""
    if month == 'Nov25':
        raw = pd.read_excel(path, sheet_name='Main Working', header=None)
        h = raw.iloc[6].tolist(); h[0] = 'Barcode'
        df = raw.iloc[7:].copy(); df.columns = [str(x).strip() for x in h]
        cnt = Counter(); nc = []
        for c in df.columns:
            cnt[c] += 1; nc.append(f"{c}.dup{cnt[c]-1}" if cnt[c] > 1 else c)
        df.columns = nc
    elif month == 'Dec25':
        df = pd.read_excel(path, sheet_name='Main Working', header=5)
        df.columns = [str(c).strip() for c in df.columns]
    else:
        df = pd.read_excel(path, sheet_name='Main Working', header=4)
        df.columns = [str(c).strip() for c in df.columns]

    df['month']        = month
    df['Success']      = pd.to_numeric(df['Success'], errors='coerce')
    df['Availability'] = pd.to_numeric(df.get('Availability', np.nan), errors='coerce')
    tpcq = df['TP CQ'].astype(str).str.strip().str.upper()
    df['rule_pred'] = tpcq.isin(['TP', 'FP']).astype(int)

    # Reference data lookups
    ref = pd.read_excel(path, sheet_name='Refrence Data', header=0)
    ref.columns = [str(c).strip() for c in ref.columns]
    ref = ref.drop_duplicates('Subcat_ASP Key', keep='first')
    for c in ref.select_dtypes(include='number').columns:
        ref[c] = pd.to_numeric(ref[c], errors='coerce').fillna(0)
    ref_d = ref.set_index('Subcat_ASP Key').to_dict('index')
    keys = df['Subcat ASP Key'].fillna('').astype(str).str.strip()
    def vc(col, fb=0.0):
        return np.array([float(ref_d.get(k, {}).get(col, fb) or fb) for k in keys])

    # Raw score columns
    for col in ['Vel\n Score', 'Vel Floor', 'Benchmark', 'Sat\n Score', 'Launch\n Score',
                'CQ\n SCORE', 'NMI\n Unit%', 'NMI\n SKU%', 'CH\n SCORE', 'Return\n Score',
                'Mon\n Score', 'SP\n SCORE', 'Launch Success\n %']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0) if col in df.columns else 0.0

    # Engineered features
    df['rtv_binary']  = (df.get('RTV Terms', pd.Series('')).astype(str).str.lower() == 'yes').astype(int)
    vfn = df['Vel Floor'].replace(0, np.nan)
    df['vel_ratio']   = (df['Benchmark'] / vfn).clip(0, 5).fillna(0)
    df['vel_ratio_sq']= df['vel_ratio'] ** 2
    df['vel_gap']     = (df['Benchmark'] - df['Vel Floor']).fillna(0)
    df['above_floor'] = (df['Benchmark'] >= df['Vel Floor']).astype(int)
    df['launch_pct']  = df['Launch Success\n %']
    df['nmi_total']   = df['NMI\n Unit%'] + df['NMI\n SKU%']
    df['nmi_ratio']   = (df['NMI\n Unit%'] / df['NMI\n SKU%'].replace(0, np.nan)).clip(0, 10).fillna(0)
    df['sat_x_vel']   = df['Sat\n Score'] * df['Vel\n Score'] / 10000
    df['cq_x_launch'] = df['CQ\n SCORE'] * (df['Launch\n Score'] > 0).astype(int)
    df['cq_sq']       = (df['CQ\n SCORE'] / 100) ** 2
    df['rtv_x_mon']   = df['rtv_binary'] * df['Mon\n Score']
    df['bench_x_sat'] = df['Benchmark'] * df['Sat\n Score'] / 10000
    df['is_width']    = (df.get('Width or Depth', pd.Series('Depth', index=df.index)) == 'Width').astype(int)
    df['conc_enc']    = df.get('Conc\n Level', pd.Series('MEDIUM', index=df.index)).map(
                            {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}).fillna(1).astype(int)
    df['asp_enc']     = df.get('ASP\n Bucket', pd.Series('25+', index=df.index)).map(
                            {'0-5':0,'5-10':1,'10-15':2,'15-20':3,'20-25':4,'25+':5}).fillna(5).astype(int)
    df['asp_log']     = np.log1p(pd.to_numeric(df.get('ASP', 0), errors='coerce').fillna(0).clip(0, 500))
    df['subcat_p50mv']= vc('Sub Category P50 ROS (Moving)')
    return df

dfs = [load_month(p, m) for m, p in BACKTEST_FILES.items()]
data = pd.concat(dfs, ignore_index=True)
data = data[data['Success'].isin([0.0, 1.0])].copy()
data['Success'] = data['Success'].astype(int)
data = data[data['Availability'] > 0.70].reset_index(drop=True)

print(f'Dataset: {len(data)} SKUs | {data["month"].nunique()} months | '
      f'{data["Success"].mean():.1%} success rate')
data.groupby('month')['Success'].agg(['count','mean']).rename(
    columns={'count':'N','mean':'Success Rate'}).round(3)


## 4. Model Definition

In [ ]:
# Logistic Regression with L2 regularisation on scaled features
# class_weight='balanced' handles label imbalance automatically
LR_MODEL = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(
                   max_iter=3000,
                   C=0.1,              # strong L2 regularisation — prevents overfitting on small months
                   penalty='l2',
                   solver='lbfgs',
                   class_weight='balanced',
                   random_state=42
               ))
])
print('Model ready:', LR_MODEL)


## 5. Leave-One-Month-Out (LOMO) Evaluation

Train on 4 months → test on the held-out month. Repeated for all 5 months.  
This mirrors real deployment: the model always predicts a month it has never seen.


In [ ]:
import copy

lomo_metrics  = {}   # month -> metric dict
lomo_coefs    = {}   # month -> Series(feature -> signed coefficient)
oof_true      = []
oof_proba     = []

for test_month in MONTHS_ORDERED:
    train_mask = data['month'] != test_month
    test_mask  = data['month'] == test_month

    X = data[BASE_FEATURES].fillna(0)
    y = data['Success']

    model = copy.deepcopy(LR_MODEL)
    model.fit(X[train_mask], y[train_mask])

    proba = model.predict_proba(X[test_mask])[:, 1]
    preds = (proba >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y[test_mask], proba)
    pc, rc, _   = precision_recall_curve(y[test_mask], proba)

    lomo_metrics[test_month] = {
        'N':         int(test_mask.sum()),
        'Accuracy':  accuracy_score(y[test_mask], preds),
        'Precision': precision_score(y[test_mask], preds, zero_division=0),
        'Recall':    recall_score(y[test_mask], preds, zero_division=0),
        'F1':        f1_score(y[test_mask], preds, zero_division=0),
        'AUC':       roc_auc_score(y[test_mask], proba),
        'AP':        average_precision_score(y[test_mask], proba),
        'fpr': fpr, 'tpr': tpr, 'pc': pc, 'rc': rc,
    }

    # Signed coefficients (on scaled features)
    coefs = model.named_steps['lr'].coef_[0]
    lomo_coefs[test_month] = pd.Series(coefs,
        index=[FEAT_LABELS[f] for f in BASE_FEATURES])

    oof_true.extend(y[test_mask].tolist())
    oof_proba.extend(proba.tolist())

oof_true  = np.array(oof_true)
oof_proba = np.array(oof_proba)

# Rule-based baseline
rule_metrics = {}
for mo in MONTHS_ORDERED:
    sub = data[data['month'] == mo]
    y_r, yp_r = sub['Success'], sub['rule_pred']
    rule_metrics[mo] = {
        'Accuracy':  accuracy_score(y_r, yp_r),
        'Precision': precision_score(y_r, yp_r, zero_division=0),
        'Recall':    recall_score(y_r, yp_r, zero_division=0),
        'F1':        f1_score(y_r, yp_r, zero_division=0),
    }

print('LOMO complete.')


## 6. Results Summary

In [ ]:
summary = pd.DataFrame({
    mo: {
        'N':                  lomo_metrics[mo]['N'],
        'LR Accuracy':        lomo_metrics[mo]['Accuracy'],
        'LR Precision':       lomo_metrics[mo]['Precision'],
        'LR Recall':          lomo_metrics[mo]['Recall'],
        'LR F1':              lomo_metrics[mo]['F1'],
        'LR AUC':             lomo_metrics[mo]['AUC'],
        'Rule Accuracy':      rule_metrics[mo]['Accuracy'],
        'Δ Accuracy vs Rule': lomo_metrics[mo]['Accuracy'] - rule_metrics[mo]['Accuracy'],
    }
    for mo in MONTHS_ORDERED
}).T

# Add mean row
summary.loc['Mean'] = summary.mean()
summary.loc['Std']  = summary.iloc[:-1].std()
summary['N'] = summary['N'].apply(lambda x: f'{int(x)}' if x == x else '—')

summary.round(3).style\
    .background_gradient(subset=['LR Accuracy','LR AUC'], cmap='Blues')\
    .background_gradient(subset=['Δ Accuracy vs Rule'], cmap='RdYlGn', vmin=-0.05, vmax=0.15)\
    .format({
        'LR Accuracy':'{:.3f}','LR Precision':'{:.3f}','LR Recall':'{:.3f}',
        'LR F1':'{:.3f}','LR AUC':'{:.3f}','Rule Accuracy':'{:.3f}',
        'Δ Accuracy vs Rule':'{:+.3f}'
    })


## 7. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('LR Model — ROC & Precision-Recall Curves (LOMO)\n'
             'Thick line = OOF overall  |  Thin lines = per month  |  ◆ = Rule-based operating point',
             fontsize=12, fontweight='bold')

# OOF overall curves
fpr_oof, tpr_oof, _ = roc_curve(oof_true, oof_proba)
auc_oof = roc_auc_score(oof_true, oof_proba)
pc_oof, rc_oof, _   = precision_recall_curve(oof_true, oof_proba)
ap_oof = average_precision_score(oof_true, oof_proba)

# Rule operating point
tn,fp,fn,tp = confusion_matrix(data['Success'], data['rule_pred'], labels=[0,1]).ravel()
rule_fpr = fp/(fp+tn); rule_tpr = tp/(tp+fn)
rule_prec = tp/(tp+fp) if (tp+fp)>0 else 0
rule_rec  = tp/(tp+fn) if (tp+fn)>0 else 0

# ROC
ax = axes[0]
for mo in MONTHS_ORDERED:
    ax.plot(lomo_metrics[mo]['fpr'], lomo_metrics[mo]['tpr'],
            color=MONTH_COLORS[mo], alpha=0.5, linewidth=1.5,
            label=f"{mo}  AUC={lomo_metrics[mo]['AUC']:.3f}")
ax.plot(fpr_oof, tpr_oof, 'k-', linewidth=3, label=f'OOF  AUC={auc_oof:.3f}', zorder=5)
ax.plot(rule_fpr, rule_tpr, 'D', color='#E65100', markersize=13,
        markeredgecolor='white', markeredgewidth=1.5, zorder=6, label='Rule ◆')
ax.plot([0,1],[0,1],'--',color='#AAAAAA',linewidth=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold'); ax.legend(fontsize=8.5, loc='lower right')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.2, linestyle='--')

# PR
ax = axes[1]
baseline = oof_true.mean()
for mo in MONTHS_ORDERED:
    ax.plot(lomo_metrics[mo]['rc'], lomo_metrics[mo]['pc'],
            color=MONTH_COLORS[mo], alpha=0.5, linewidth=1.5,
            label=f"{mo}  AP={lomo_metrics[mo]['AP']:.3f}")
ax.plot(rc_oof, pc_oof, 'k-', linewidth=3, label=f'OOF  AP={ap_oof:.3f}', zorder=5)
ax.axhline(baseline, color='#AAAAAA', linestyle='--', linewidth=1, label=f'Baseline={baseline:.3f}')
ax.plot(rule_rec, rule_prec, 'D', color='#E65100', markersize=13,
        markeredgecolor='white', markeredgewidth=1.5, zorder=6, label='Rule ◆')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold'); ax.legend(fontsize=8.5, loc='lower left')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout(); plt.show()


## 8. Month-wise Accuracy vs Rule-based

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Month-wise Accuracy — LR vs Rule-based (LOMO)', fontweight='bold', fontsize=13)

x = np.arange(len(MONTHS_ORDERED)); w = 0.35
lr_accs   = [lomo_metrics[mo]['Accuracy']  for mo in MONTHS_ORDERED]
rule_accs = [rule_metrics[mo]['Accuracy']  for mo in MONTHS_ORDERED]

b1 = ax1.bar(x - w/2, rule_accs, w, color='#E65100', alpha=0.85, label='Rule-based', edgecolor='white')
b2 = ax1.bar(x + w/2, lr_accs,   w, color='#1565C0', alpha=0.85, label='LR Model',   edgecolor='white')
for bar, v in zip(b1, rule_accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{v:.1%}', ha='center', fontsize=9, color='#E65100', fontweight='bold')
for bar, v in zip(b2, lr_accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{v:.1%}', ha='center', fontsize=9, color='#1565C0', fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(MONTHS_ORDERED)
ax1.set_ylim(0.45, 0.82); ax1.set_ylabel('Accuracy')
ax1.axhline(np.mean(rule_accs), color='#E65100', linestyle=':', linewidth=2, alpha=0.7)
ax1.axhline(np.mean(lr_accs),   color='#1565C0', linestyle=':', linewidth=2, alpha=0.7)
ax1.legend(fontsize=10); ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.spines[['top','right']].set_visible(False)

# Lift chart
lifts = [lomo_metrics[mo]['Accuracy'] - rule_metrics[mo]['Accuracy'] for mo in MONTHS_ORDERED]
colors = ['#2E7D32' if l > 0 else '#C62828' for l in lifts]
bars = ax2.bar(MONTHS_ORDERED, lifts, color=colors, alpha=0.85, edgecolor='white')
for bar, v in zip(bars, lifts):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (0.003 if v >= 0 else -0.010),
             f'{v:+.1%}', ha='center', va='bottom' if v >= 0 else 'top',
             fontsize=10, fontweight='bold',
             color='#2E7D32' if v >= 0 else '#C62828')
ax2.axhline(0, color='black', linewidth=1.2)
ax2.set_ylabel('Accuracy Lift vs Rule')
ax2.set_title('LR Lift over Rule-based', fontweight='bold')
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout(); plt.show()
print(f'Mean LR accuracy: {np.mean(lr_accs):.1%}  (std {np.std(lr_accs):.1%})')
print(f'Mean rule accuracy: {np.mean(rule_accs):.1%}  (std {np.std(rule_accs):.1%})')
print(f'Mean lift: {np.mean(lifts):+.1%}')


## 9. Feature Importances — Signed Coefficients

In [ ]:
coef_df   = pd.DataFrame(lomo_coefs)          # features × months
mean_coef = coef_df.mean(axis=1)
std_coef  = coef_df.std(axis=1)
order     = mean_coef.abs().sort_values(ascending=False).index
coef_df   = coef_df.loc[order]
mean_coef = mean_coef[order]
std_coef  = std_coef[order]

TOP_N = 20
top_feats = order[:TOP_N].tolist()

fig, ax = plt.subplots(figsize=(11, 8))
fig.suptitle('LR Coefficients — Mean ± Std (5 LOMO folds)\n'
             'Blue = increases success probability  |  Red = decreases  |  ⚠ = sign flips across months',
             fontsize=11, fontweight='bold')
ax.set_facecolor('#FAFAFA')

y_pos  = np.arange(len(top_feats))
means  = mean_coef[top_feats].values
stds   = std_coef[top_feats].values
colors = ['#1565C0' if v >= 0 else '#C62828' for v in means]

ax.barh(y_pos, means, color=colors, alpha=0.85, edgecolor='white', height=0.65)
ax.errorbar(means, y_pos, xerr=stds, fmt='none',
            color='#333333', capsize=5, linewidth=1.8, capthick=1.8, zorder=5)
ax.axvline(0, color='black', linewidth=1.2)
ax.set_yticks(y_pos); ax.set_yticklabels(top_feats, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Coefficient (standardised features)', fontsize=10)
ax.grid(axis='x', alpha=0.25, linestyle='--')
ax.spines[['top','right']].set_visible(False)

for i, (f, v, s) in enumerate(zip(top_feats, means, stds)):
    vals = coef_df.loc[f]
    stable = (vals > 0).all() or (vals < 0).all()
    xpos = v + s + 0.005 if v >= 0 else v - s - 0.005
    ax.text(xpos, i, f'{v:+.3f}±{s:.3f}',
            va='center', fontsize=8, color=colors[i], fontweight='bold',
            ha='left' if v >= 0 else 'right')
    if not stable:
        ax.text(0, i, ' ⚠', va='center', fontsize=11, color='#FF8F00')

ax.legend(handles=[
    Patch(color='#1565C0', label='Positive → success'),
    Patch(color='#C62828', label='Negative → failure'),
    Patch(color='#FF8F00', label='⚠ sign flips across months'),
], fontsize=9, loc='lower right')

plt.tight_layout(); plt.show()


## 10. Coefficient Stability Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
fig.suptitle('LR Coefficient Values per Month\n'
             'Blue = positive  |  Red = negative  |  Orange border = sign flips',
             fontsize=11, fontweight='bold')

heat  = coef_df.loc[top_feats].values
vmax  = np.abs(heat).max()
im    = ax.imshow(heat, cmap='RdBu', aspect='auto', vmin=-vmax, vmax=vmax)

ax.set_xticks(range(len(MONTHS_ORDERED)))
ax.set_xticklabels(MONTHS_ORDERED, fontsize=10)
ax.set_yticks(range(len(top_feats)))
ax.set_yticklabels(top_feats, fontsize=9.5)

for i in range(len(top_feats)):
    for j in range(len(MONTHS_ORDERED)):
        v = heat[i, j]
        ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(v) > vmax * 0.55 else 'black', fontweight='bold')

for i, f in enumerate(top_feats):
    vals = coef_df.loc[f]
    if not ((vals > 0).all() or (vals < 0).all()):
        ax.add_patch(plt.Rectangle((-0.5, i-0.5), len(MONTHS_ORDERED), 1,
                     fill=False, edgecolor='#FF8F00', linewidth=2.5, zorder=5))

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Coefficient value')
plt.tight_layout(); plt.show()

# Print stability summary
print('Sign stability for top features:')
for f in top_feats:
    vals = coef_df.loc[f]
    stable = (vals > 0).all() or (vals < 0).all()
    print(f'  {f:<20} {"✓ stable":>12}  mean={vals.mean():+.3f}  std={vals.std():.3f}')


## 11. Confusion Matrix (OOF — all months combined)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Confusion Matrices — OOF Combined (791 SKUs)', fontweight='bold', fontsize=12)

oof_preds = (oof_proba >= 0.5).astype(int)
rule_preds = data['rule_pred'].values

for ax, (title, y_pred, color) in zip(axes, [
    ('Rule-based',  rule_preds, '#E65100'),
    ('LR Model',    oof_preds,  '#1565C0'),
]):
    cm = confusion_matrix(data['Success'], y_pred, labels=[0,1])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Fail','Success'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(data['Success'], y_pred)
    f1  = f1_score(data['Success'], y_pred, zero_division=0)
    ax.set_title(f'{title}\nAcc={acc:.3f}  F1={f1:.3f}', fontweight='bold', color=color)

plt.tight_layout(); plt.show()


## 12. Predicted Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.suptitle('Predicted Success Probability — OOF\n'
             'Well-separated distributions = model has discriminative power', fontweight='bold')

success_mask = data['Success'] == 1
ax.hist(oof_proba[~success_mask], bins=30, alpha=0.65, color='#C62828',
        label='Actual Fail (0)', density=True, edgecolor='white')
ax.hist(oof_proba[success_mask],  bins=30, alpha=0.65, color='#1565C0',
        label='Actual Success (1)', density=True, edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.set_xlabel('Predicted Probability of Success', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.25, linestyle='--')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


## 13. Production Scoring Function

Train the final model on **all 5 months** and use it to score new SKUs.


In [ ]:
# Train on all data
import copy
final_model = copy.deepcopy(LR_MODEL)
X_all = data[BASE_FEATURES].fillna(0)
y_all = data['Success']
final_model.fit(X_all, y_all)

def score_skus(df_new):
    """
    Score new SKUs using the trained LR model.
    df_new must have all columns needed for feature engineering.
    Returns the input df with two new columns:
      - success_proba : probability of success (0-1)
      - success_pred  : binary prediction at 0.5 threshold
    """
    X_new = df_new[BASE_FEATURES].fillna(0)
    proba = final_model.predict_proba(X_new)[:, 1]
    result = df_new.copy()
    result['success_proba'] = proba
    result['success_pred']  = (proba >= 0.5).astype(int)
    return result

# Quick sanity check on training data
check = score_skus(data)
print(f'In-sample accuracy: {accuracy_score(y_all, check["success_pred"]):.1%}')
print(f'Predicted success rate: {check["success_proba"].mean():.1%}')
print('\nSample predictions:')
check[['month','success_proba','success_pred','Success']].head(10)
